# Microsoft Foundry Local 
## Running Large Language Models Locally with Foundry Local

<img src="https://github.com/retkowsky/foundry-local/blob/main/foundrylocal.jpg?raw=true">

This notebook is a **hands-on demonstration of running and interacting with Large Language Models (LLMs) locally** using **Foundry Local** and an **OpenAI-compatible API**.  
It showcases how to discover, load, and interact with models entirely on-device cloud services.

The notebook is intended as a **practical reference and demo** for:
- Offline / disconnected environments
- Edge or on-device AI scenarios
- Enterprise and customer demonstrations
- Prompt engineering and API pattern exploration

---

## 📦 Environment & Setup

- Python **3.12**
- Common data and utility libraries: `pandas`, `matplotlib`, `requests`, `statistics`
- Foundry Local runtime managed via `FoundryLocalManager`
- Local OpenAI-compatible endpoint (no API key required for local usage)

The notebook starts the Foundry Local service, validates its status, and exposes:
- Service URI and API endpoint
- Local model cache location

---

## 🧩 Model Catalog & Management

- Lists all available local model variants from the Foundry catalog
- Converts model metadata into a clean `pandas.DataFrame` including:
  - Alias and model ID
  - Device type (CPU / GPU)
  - Execution provider
  - Model size
  - License and supported tasks
  - Tool-calling support
- Supports filtering models by alias (e.g. `phi-4`)
- Demonstrates loading a specific model into memory

---

## 💬 Chat Completions (OpenAI-Compatible)

### Synchronous Chat Completion
- Sends a prompt and receives the full response in one call
- Illustrates basic question–answer usage
- Highlights local response behavior (e.g. missing token usage metadata)

### Streaming Chat Completion
- Streams tokens incrementally as they are generated
- Measures throughput (tokens per second)
- Ideal for real-time UIs and low-latency user experiences

---

## 🎛️ Prompt Engineering & Output Control

### System Prompts
- Control assistant behavior, tone, and format
- Enforce structured outputs (e.g. JSON-only responses)

### Examples Included
- **Structured JSON extraction** (entity extraction from text)
- **Persona-based responses** (e.g. friendly data scientist, playful explanations)
- **Temperature tuning** for deterministic vs. creative outputs

---

## 🔁 Multi-Turn Conversations

- Demonstrates how to maintain conversation context
- Uses a shared `messages` array across multiple turns
- Example: concise math tutor answering follow-up questions
- Highlights that context is managed client-side

---

## 🌐 Direct REST API Usage

- Calls the Foundry Local REST API directly using `requests`
- Bypasses the OpenAI SDK entirely
- Shows raw request / response structure
- Confirms OpenAI API compatibility at the HTTP level

> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/ai-foundry/foundry-local/reference/reference-sdk?pivots=programming-language-python)

## Author

| Field | Details |
| --- | --- |
| Name | Serge Retkowsky |
| Email | serge.retkowsky@microsoft.com |
| LinkedIn | https://www.linkedin.com/in/serger/ |
| Medium publications | https://medium.com/@sergems18/ |

## Setup

In [1]:
import datetime
import GPUtil
import json
import matplotlib.pyplot as plt
import os
import pandas as pd
import platform
import psutil
import requests
import sys
import time
import statistics

from foundry_local import FoundryLocalManager
from openai import OpenAI

In [2]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 25-Feb-2026 12:46:31


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(
    f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical"
)

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(
            f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)"
        )
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 23.5 GB
- RAM used:      63.2%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  894.6 GB (94.0%)
- Free:  57.1 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 3,666 MB (90%)
- VRAM Free  : 270 MB
- GPU Load   : 30%
- Temperature: 61.0 °C


In [5]:
manager = FoundryLocalManager()
manager.start_service()

print(f"Service running : {manager.is_service_running()}")
print(f"Service URI     : {manager.service_uri}")
print(f"Endpoint (v1)   : {manager.endpoint}")
print(f"Cache location  : {manager.get_cache_location()}")

Service running : True
Service URI     : http://127.0.0.1:55311
Endpoint (v1)   : http://127.0.0.1:55311/v1
Cache location  : C:\models


In [6]:
os.listdir(manager.get_cache_location())

['Microsoft']

In [7]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

[]

## Helper

In [8]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([{
        "alias": m.alias,
        "id": m.id,
        "device": m.device_type.value,
        "provider": m.execution_provider,
        "size_mb": m.file_size_mb,
        "tools_suport": m.supports_tool_calling,
        "license": m.license,
        "task": m.task,
    } for m in models])

## Models

In [9]:
catalog = manager.list_catalog_models()
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 80


,alias,id,device,provider,size_mb,tools_suport,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
4,phi-3.5-mini,Phi-3.5-mini-instruct-cuda-gpu:1,GPU,CUDAExecutionProvider,2181,False,MIT,chat-completion
...,...,...,...,...,...,...,...,...
75,qwen2.5-7b,qwen2.5-7b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,6307,True,apache-2.0,chat-completion
76,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-cuda-gpu:2,GPU,CUDAExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
77,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-generic-cpu:2,CPU,CPUExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
78,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,False,MIT,chat-completion


In [10]:
# Filter by alias keyword
df_catalog[df_catalog["alias"].str.contains("phi-4", case=False, na=False)]

,alias,id,device,provider,size_mb,tools_suport,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
40,phi-4-mini-reasoning,Phi-4-mini-reasoning-cuda-gpu:3,GPU,CUDAExecutionProvider,3225,False,MIT,chat-completion
41,phi-4-mini-reasoning,Phi-4-mini-reasoning-openvino-gpu:2,GPU,OpenVINOExecutionProvider,2532,False,MIT,chat-completion
42,phi-4-mini-reasoning,Phi-4-mini-reasoning-generic-gpu:3,GPU,WebGpuExecutionProvider,3225,False,MIT,chat-completion
43,phi-4-mini-reasoning,Phi-4-mini-reasoning-generic-cpu:3,CPU,CPUExecutionProvider,4628,False,MIT,chat-completion
56,phi-4-mini,Phi-4-mini-instruct-cuda-gpu:5,GPU,CUDAExecutionProvider,3686,True,MIT,chat-completion
57,phi-4-mini,phi-4-mini-instruct-openvino-gpu:2,GPU,OpenVINOExecutionProvider,2205,True,MIT,chat-completion


### Let's use the phi-4-mini model
https://techcommunity.microsoft.com/blog/educatordeveloperblog/welcome-to-the-new-phi-4-models---microsoft-phi-4-mini--phi-4-multimodal/4386037

In [11]:
manager.download_model("Phi-4-mini")  # it will load the best model according to your infra

FoundryModelInfo(alias=phi-4-mini, id=Phi-4-mini-instruct-cuda-gpu:5, execution_provider=CUDAExecutionProvider, device_type=GPU, file_size=3686 MB, license=MIT)

In [12]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

['Phi-4-mini-instruct-cuda-gpu-5']

In [13]:
# Load a model into memory
model_info = manager.load_model("Phi-4-mini")
print(f"Loaded: {model_info.alias} => {model_info.id}")

Loaded: phi-4-mini => Phi-4-mini-instruct-cuda-gpu:5


In [14]:
info = manager.get_model_info("Phi-4-mini")
print(f"Running on: {info.execution_provider}")
print(f"Device:     {info.device_type}")

Running on: CUDAExecutionProvider
Device:     GPU


In [15]:
# Initialize the OpenAI client
client = OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key  # Not required for local usage
)

## Synchronous Chat Completion

The simplest way to get a response — send a message, get the full response back at once.

In [16]:
response = client.chat.completions.create(
    model=model_info.id,
    messages=[{
        "role": "user",
        "content": "What is Pi? Explain in 2 sentences."
    }])

print("🤖 Response:")
print(response.choices[0].message.content)

🤖 Response:
Pi (π) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning it cannot be exactly expressed as a simple fraction, and its decimal representation never ends or repeats.


In [17]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.54",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning it cannot be exactly expressed as a simple fraction, and its decimal representation never ends or repeats.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning it cannot be exactly expressed as a simple fraction, and its decimal representation never ends or repeats.",
        "tool_calls": []
      }
    }

## Streaming Chat Completion

Streaming returns tokens incrementally as they're generated — great for real-time UIs and lower perceived latency.

In [18]:
print("Streaming response:\n")

start = time.time()

stream = client.chat.completions.create(
    model=model_info.id,
    messages=[{
        "role": "user",
        "content": "Explain quantum computing in 5 lines."
    }],
    stream=True)

token_count = 0

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)
        token_count += 1

elapsed = time.time() - start

print(f"\n\n⏱️  {token_count} tokens in {elapsed:.2f}s "
      f"({token_count/elapsed:.1f} tokens/sec)")

Streaming response:

Quantum computing harnesses the principles of quantum mechanics to process information in ways that traditional computers cannot. It uses quantum bits, or qubits, which can exist in multiple states simultaneously, enabling parallel computation. Quantum entanglement and superposition allow for the solving of complex problems more efficiently. Quantum computers have the potential to revolutionize fields like cryptography, optimization, and simulation of molecular structures. However, they are still in the experimental stage, facing challenges in error correction and scalability. Researchers are actively working to overcome these hurdles to realize practical quantum computing applications.

⏱️  111 tokens in 16.35s (6.8 tokens/sec)


In [19]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.54",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning it cannot be exactly expressed as a simple fraction, and its decimal representation never ends or repeats.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning it cannot be exactly expressed as a simple fraction, and its decimal representation never ends or repeats.",
        "tool_calls": []
      }
    }

## System Prompts

Use system messages to control the model's behavior, persona, and output format.

In [20]:
# Example 1: JSON output mode
response = client.chat.completions.create(
    model=model_info.id,
    messages=[{
        "role":
        "system",
        "content":
        "You are a helpful data extraction assistant. "
        "Always respond with valid JSON only, no markdown."
    }, {
        "role":
        "user",
        "content":
        "Extract the key entities from this sentence: "
        "'Marie Curie won the Nobel Prize in Physics in 1903 in Stockholm.'"
    }],
    temperature=0.1  # Low temperature for deterministic output
)

print(response.choices[0].message.content)

{"entities": [{"entity": "Marie Curie", "type": "PERSON"}, {"entity": "Nobel Prize", "type": "EVENT"}, {"entity": "Physics", "type": "FIELD"}, {"entity": "1903", "type": "DATE"}, {"entity": "Stockholm", "type": "LOCATION"}]}


In [21]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.56",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"entities\": [{\"entity\": \"Marie Curie\", \"type\": \"PERSON\"}, {\"entity\": \"Nobel Prize\", \"type\": \"EVENT\"}, {\"entity\": \"Physics\", \"type\": \"FIELD\"}, {\"entity\": \"1903\", \"type\": \"DATE\"}, {\"entity\": \"Stockholm\", \"type\": \"LOCATION\"}]}",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "{\"entities\": [{\"entity\": \"Marie Curie\", \"type\": \"PERSON\"}, {\"entity\": \"Nobel Prize\", \"type\": \"EVENT\"}, {\"entity\": \"Physics\", \"type\": \"FIELD\"}, {\"entity\": \"1903\", \"type\": \"DATE\"}, {\"entity\": \"Stockholm\", \"type\": \"LOCATION\"}]}",
        "tool_calls": []
      }
    }
  ],
  "created": 1772020143,

In [22]:
# Example 2: Persona / role-play
response = client.chat.completions.create(
    model=model_info.id,
    messages=[{
        "role":
        "system",
        "content":
        "You are a friendly datascientist that explains things "
        "using simple text. Keep answers short and fun."
    }, {
        "role":
        "user",
        "content":
        "What is an object detection model in computer vision?"
    }],
    temperature=0.7  # Higher temperature for creativity
)

print(response.choices[0].message.content)

An object detection model in computer vision is like a smart camera that can look at an image and say, "Hey, I see a dog, a car, and a tree in there!" It uses special computer algorithms to find and label different objects in pictures or videos. Think of it as a video game character that can spot and identify all the different items in a scene.


In [23]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.57",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "An object detection model in computer vision is like a smart camera that can look at an image and say, \"Hey, I see a dog, a car, and a tree in there!\" It uses special computer algorithms to find and label different objects in pictures or videos. Think of it as a video game character that can spot and identify all the different items in a scene.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "An object detection model in computer vision is like a smart camera that can look at an image and say, \"Hey, I see a dog, a car, and a tree in there!\" It uses special computer algorithms to find and label different objects in pictures or videos. Think o

## Multi-Turn Conversations

Maintain conversation context by passing the full message history. The model has no memory between calls — context lives in the messages array.

In [24]:
# Simulate a multi-turn conversation
conversation = [
    {
        "role": "system",
        "content": "You are a helpful math tutor. Be concise."
    },
]

questions = [
    "What is a derivative in calculus?",
    "Can you give me a simple example?",
    "What about the chain rule?",
]

for q in questions:
    # Add user message
    conversation.append({"role": "user", "content": q})

    # Get response
    response = client.chat.completions.create(model=model_info.id,
                                              messages=conversation,
                                              max_tokens=2000)

    assistant_msg = response.choices[0].message.content

    # Add assistant response to history
    conversation.append({"role": "assistant", "content": assistant_msg})

    print(f"👤 User: {q}")
    print(f"🤖 Assistant: {assistant_msg}")
    print("-" * 60)

print(f"\n📝 Conversation history: {len(conversation)} messages")

👤 User: What is a derivative in calculus?
🤖 Assistant: A derivative in calculus represents the rate at which a function is changing at any given point. It is a measure of how a function's output value changes as its input value changes. Mathematically, the derivative of a function f(x) at a point x is defined as the limit of the average rate of change of the function over an interval as the interval approaches zero. It is denoted as f'(x) or df/dx. The derivative provides valuable information about the behavior of functions, such as their slopes, rates of change, and critical points.
------------------------------------------------------------
👤 User: Can you give me a simple example?
🤖 Assistant: Sure! Let's consider the function f(x) = x^2. To find the derivative of this function, we'll use the power rule, which states that if f(x) = x^n, then f'(x) = n*x^(n-1).

Applying the power rule to our function, we get:

f'(x) = 2*x^(2-1) = 2*x

So, the derivative of f(x) = x^2 is f'(x) = 2x.

In [25]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.60",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The chain rule is a fundamental rule in calculus used to find the derivative of a composite function. It states that if you have two functions, say f(x) and g(x), and you want to find the derivative of their composition, denoted as (f \u2218 g)(x) = f(g(x)), then the derivative of the composite function is given by the product of the derivatives of the individual functions, i.e., (f \u2218 g)'(x) = f'(g(x)) * g'(x).\n\nIn simpler terms, the chain rule allows you to differentiate a function that is composed of other functions by multiplying the derivative of the outer function (evaluated at the inner function) by the derivative of the inner function.\n\nHere's an example to illustrate the chain rule:\n\nLet's say we have two functions, f(u) = u^2 and g(x) = 3x + 1. We want to find the derivative of the composite function (f \u2218 g

## Direct REST API with `requests`

You can also call the Foundry Local REST API directly without the OpenAI SDK.

In [26]:
# Build the request manually
url = manager.endpoint + "/chat/completions"

payload = {
    "model": model_info.id,
    "messages": [{
        "role": "user",
        "content": "What is the speed of light?"
    }],
    "temperature": 0.7,
    "max_tokens": 2000
}

headers = {"Content-Type": "application/json"}

resp = requests.post(url, headers=headers, data=json.dumps(payload))
data = resp.json()

print(f"Status: {resp.status_code}")
print(f"Model: {data['model']}")
print(f"{data['choices'][0]['message']['content']}")

Status: 200
Model: Phi-4-mini-instruct-cuda-gpu:5
The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s) or about 186,282 miles per second (mi/s). This constant is denoted by the symbol "c" and is a fundamental constant of nature. The speed of light is crucial in the field of physics, particularly in the theory of relativity developed by Albert Einstein. It is the maximum speed at which all energy, matter, and information in the universe can travel.


In [27]:
print(resp)

<Response [200]>


In [28]:
print(data)

{'model': 'Phi-4-mini-instruct-cuda-gpu:5', 'choices': [{'delta': {'role': 'assistant', 'content': 'The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s) or about 186,282 miles per second (mi/s). This constant is denoted by the symbol "c" and is a fundamental constant of nature. The speed of light is crucial in the field of physics, particularly in the theory of relativity developed by Albert Einstein. It is the maximum speed at which all energy, matter, and information in the universe can travel.', 'tool_calls': []}, 'message': {'role': 'assistant', 'content': 'The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s) or about 186,282 miles per second (mi/s). This constant is denoted by the symbol "c" and is a fundamental constant of nature. The speed of light is crucial in the field of physics, particularly in the theory of relativity developed by Albert Einstein. It is the maximum speed at which all energy, matter, and inform

# What We Learned?

## Overview

This notebook walks through how to use **Microsoft Foundry Local** to run AI models entirely on a Windows machine, powered by ONNX Runtime, while maintaining an OpenAI-compatible API surface.

The end-to-end workflow covers service setup, model discovery and filtering, local caching, model loading and inference, and cleanup with memory management.

---

## 🚀 Why Foundry Local Matters

Hands-on execution revealed several compelling advantages:

- **Local-only inference** — All prompts and data remain on the machine with no network calls required.
- **Zero runtime cost** — Once models are downloaded, inference runs without any API or token charges.
- **Low latency and offline usage** — No cloud round-trips; performance depends solely on local hardware.
- **Cloud-compatible developer experience** — The API follows the OpenAI specification, so existing SDKs and code work by simply switching the endpoint.
- **Optimized for consumer hardware** — Models are pre-optimized via Microsoft Olive for CPU, CUDA GPU, OpenVINO, and WebGPU backends.

---

## 🖥️ Local Environment Insights

The notebook confirmed that Foundry Local:

- Runs smoothly on **Windows 11**.
- Works with **CPU-only setups** as well as mid-range GPUs.
- Automatically selects the **best model variant** based on available hardware.
- Exposes a local REST endpoint on `127.0.0.1` with a dynamically assigned port.
- Stores models in a **local cache** (e.g., `C:\models`).

---

## 🧠 Model Catalog Exploration

Key takeaways from inspecting the catalog:

The catalog contains **multiple variants per model**, each optimized for a different hardware target. Supported model families include Phi-4, Phi-4-mini, Phi-4-mini-reasoning, Phi-3 and Phi-3.5, Qwen 2.5, Mistral, DeepSeek, Whisper (speech-to-text), and gpt-oss-20b.

Models differ along several dimensions: device type (CPU or GPU), execution provider (CPU, CUDA, OpenVINO, WebGPU), size (from a few hundred megabytes to over 12 GB), and task type (chat completion, automatic speech recognition).

Through filtering and visualization, it was straightforward to identify CPU-only models, spot tool-calling–enabled variants, and compare sizes and providers at a glance.

---

## 📦 Cache and Model Lifecycle

Several observations on model management:

- Downloaded models are **cached locally** and reused across sessions.
- Loading a model is **fast once cached**, with no re-download needed.
- **Multiple models** can be loaded simultaneously, memory permitting.

---

## 🤖 Running Inference with the OpenAI SDK

One of the most striking takeaways is how seamless local inference feels:

- Foundry Local exposes an **OpenAI-compatible endpoint**, meaning the standard `openai` Python SDK works unchanged.
- The model ID must be passed as a **string**, not as a model object.
- Switching between models is as simple as changing the `model` parameter.

Example results confirmed that **Phi-4** ran on GPU via `CUDAExecutionProvider`, while **gpt-oss-20b** ran on CPU — and both produced responses identical in structure to cloud-based chat completions.

---

## 🧹 Cleanup and Resource Management

Finally, Foundry Local provides clean resource controls:

- Models can be **unloaded from memory** without deleting them from cache.
- Unloading **frees RAM and VRAM immediately**.
- Cached models remain available for **instant reload** whenever needed.

> Go to the next notebook